# 34 · Rewind session history

Rewind restores state and the active conversation to before a selected invocation. ADK keeps an audit trail and appends a rewind event; it does not delete stored events or reverse external side effects.

**Runtime:** ADK 2.10.0 on `aadhi`, existing `dataengenv` kernel and ADC.

**Cost:** 2 small Lite calls. Synthetic ecommerce data only. Run cells from top to bottom; each notebook creates its own sessions.


In [ ]:
from pathlib import Path
import sys
# Works when Jupyter starts in GoogleADK or its parent folder.
root = Path.cwd() if (Path.cwd() / "course.py").exists() else Path.cwd() / "GoogleADK"
if not (root / "course.py").exists():
    raise RuntimeError("Open this notebook with GoogleADK as the working directory.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from course import (ROOT, CONFIG, MODEL, PROJECT, LOCATION, SKIP_SSL, DATA,
                    llm, gemini, make_lab, ask, final_text, token_usage,
                    get_order, get_product, get_customer, types)
print(f"Project={PROJECT} | location={LOCATION} | model={MODEL} | TLS bypass={SKIP_SSL}")


In [ ]:
agent = llm("rewind_shop", "Acknowledge the supplied order ID briefly.", output_key="last_order_response")
lab = await make_lab(agent)
await ask(lab, "My order is O1001.")
events = await ask(lab, "Correction: my order is O1002.")
invocation_id = events[0].invocation_id
before = len(lab.session.events)
await lab.runner.rewind_async(user_id=lab.user_id, session_id=lab.session.id,
                              rewind_before_invocation_id=invocation_id)
restored = await lab.runner.session_service.get_session(app_name=lab.runner.app_name,
    user_id=lab.user_id, session_id=lab.session.id)
print("Audit event counts before/after:", before, len(restored.events))
assert restored.events[-1].actions.rewind_before_invocation_id == invocation_id
assert "O1001" in restored.state["last_order_response"]
assert "O1002" not in restored.state["last_order_response"]
print("Restored response:", restored.state["last_order_response"])


## Try it

Inspect the remaining user messages. Combine this with SQLite persistence for a recoverable conversation editor.


## Reference

[Official ADK documentation](https://adk.dev/sessions/rewind/). Shared configuration: `config.json`. No environment activation or login cells are needed.
